In [1]:
# Step one is to assign a state, I am selecting TypeDict
from typing import TypedDict, Literal, Optional, List, Dict, Any

StructureSource = Literal["pdb", "swiss_model", "alphafold", "alphafill", "none"]

class UserQuery(TypedDict, total=False):
    accession: str
    organism: Optional[str]
    user_request: str
    preferred_source: Optional[str]

class StructureCandidate(TypedDict, total=False):
    source: StructureSource
    structure_id: Optional[str]
    url: Optional[str]
    local_path: Optional[str]

    # metrics
    coverage: Optional[float]
    resolution: Optional[float]      # lower is better for experimental structures
    confidence: Optional[float]      # pLDDT or analogous model confidence
    method: Optional[str]            # xray, cryo-em, homology, AF2, AlphaFill, etc.
    has_ligand_context: Optional[bool]
    sequence_length: Optional[int]
    aligned_length: Optional[int]

    # bookkeeping
    available: bool
    validator_notes: List[str]
    raw_metadata: Dict[str, Any]

class GraphState(TypedDict, total=False):
    query: UserQuery
    messages: List[dict]

    candidates: List[StructureCandidate]
    best_candidate: Optional[StructureCandidate]

    retrieval_complete: bool
    validation_complete: bool
    selection_complete: bool

    route_decision: Optional[str]
    rationale: Optional[str]
    errors: List[str]

In [ ]:
# Database Tools
import requests
from typing import List

def fetch_pdb_candidates(accession: str) -> List[StructureCandidate]:
    # placeholder
    # query RCSB / mapping layer
    return []

def fetch_swiss_model_candidate(accession: str) -> List[StructureCandidate]:
    return []

def fetch_alphafold_candidate(accession: str) -> List[StructureCandidate]:
    return []

def fetch_alphafill_candidate(accession: str) -> List[StructureCandidate]:
    return []

In [ ]:
# Retrieval node
def retrieve_candidates(state: GraphState) -> GraphState:
    accession = state["query"]["accession"]
    errors = state.get("errors", [])
    candidates = []

    for fn in [
        fetch_pdb_candidates,
        fetch_swiss_model_candidate,
        fetch_alphafold_candidate,
        fetch_alphafill_candidate,
    ]:
        try:
            result = fn(accession)
            if result:
                candidates.extend(result)
        except Exception as e:
            errors.append(f"{fn.__name__} failed: {e}")

    return {
        **state,
        "candidates": candidates,
        "retrieval_complete": True,
        "errors": errors,
    }

In [ ]:
# Validator node
def validate_candidates(state: GraphState) -> GraphState:
    validated = []

    for c in state.get("candidates", []):
        notes = list(c.get("validator_notes", []))

        source = c["source"]
        coverage = c.get("coverage")
        resolution = c.get("resolution")
        confidence = c.get("confidence")

        if coverage is None:
            notes.append("Missing coverage")
        if source == "pdb" and resolution is None:
            notes.append("Missing experimental resolution")
        if source in ("alphafold", "alphafill") and confidence is None:
            notes.append("Missing model confidence")

        c["validator_notes"] = notes
        c["available"] = True
        validated.append(c)

    return {
        **state,
        "candidates": validated,
        "validation_complete": True,
    }

In [ ]:
# Ranking Classifier
def score_candidate(c: StructureCandidate) -> float:
    score = 0.0
    source = c["source"]

    coverage = c.get("coverage") or 0.0
    resolution = c.get("resolution")
    confidence = c.get("confidence") or 0.0
    has_ligand = c.get("has_ligand_context") or False

    score += coverage * 50

    if source == "pdb":
        score += 40
        if resolution is not None:
            score += max(0, 10 - resolution * 2)
    elif source == "alphafill":
        score += 30
        if has_ligand:
            score += 10
        score += confidence / 10
    elif source == "swiss_model":
        score += 20
        score += confidence / 10
    elif source == "alphafold":
        score += 15
        score += confidence / 10

    return score

def rank_candidates(state: GraphState) -> GraphState:
    candidates = state.get("candidates", [])
    ranked = sorted(candidates, key=score_candidate, reverse=True)

    return {
        **state,
        "candidates": ranked,
        "route_decision": "ranked" if ranked else "none_found",
    }

In [ ]:
# Choose the best structure based on pre-set parameters
def choose_best_structure(state: GraphState) -> GraphState:
    ranked = state.get("candidates", [])
    if not ranked:
        return {
            **state,
            "best_candidate": None,
            "selection_complete": True,
            "rationale": "No valid candidate structures found."
        }

    best = ranked[0]
    rationale = (
        f"Selected {best['source']} structure "
        f"{best.get('structure_id')} based on highest combined score from "
        f"coverage, resolution/confidence, and ligand-context availability."
    )

    return {
        **state,
        "best_candidate": best,
        "selection_complete": True,
        "rationale": rationale,
    }

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="phi3")

def explain_selection(state: GraphState) -> GraphState:
    best = state.get("best_candidate")
    if not best:
        return state

    prompt = f"""
You are explaining protein structure selection for a proteomics pipeline.

User request:
{state['query'].get('user_request', '')}

Best candidate:
{best}

Other candidates:
{state.get('candidates', [])[:5]}

Explain in 4-6 sentences why this structure was selected.
"""
    response = llm.invoke(prompt)

    return {
        **state,
        "rationale": response.content
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

def route_after_validation(state: GraphState):
    if not state.get("candidates"):
        return "fallback_response"
    return "rank_candidates"

builder = StateGraph(GraphState)

builder.add_node("retrieve_candidates", retrieve_candidates)
builder.add_node("validate_candidates", validate_candidates)
builder.add_node("rank_candidates", rank_candidates)
builder.add_node("choose_best_structure", choose_best_structure)
builder.add_node("explain_selection", explain_selection)

builder.add_node(
    "fallback_response",
    lambda state: {
        **state,
        "rationale": "No structure candidates were found from PDB, Swiss-Model, AlphaFold, or AlphaFill."
    }
)

builder.add_edge(START, "retrieve_candidates")
builder.add_edge("retrieve_candidates", "validate_candidates")
builder.add_conditional_edges(
    "validate_candidates",
    route_after_validation,
    {
        "fallback_response": "fallback_response",
        "rank_candidates": "rank_candidates",
    }
)
builder.add_edge("rank_candidates", "choose_best_structure")
builder.add_edge("choose_best_structure", "explain_selection")
builder.add_edge("explain_selection", END)
builder.add_edge("fallback_response", END)

graph = builder.compile()